In [ ]:
import os
import cv2
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.models as models
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm



# 1. INFERENCE CONFIGURATION




In [ ]:
TEST_DIR = '/kaggle/input/competitions/aptos2019-blindness-detection/test_images'
MODEL_WEIGHTS = '/kaggle/input/datasets/hridhdeeshjain15/finalley/best_densenet_qwk.pth' 

IMAGE_SIZE = 224       
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# 2. PREPROCESSING & TRANSFORMS


In [ ]:


def crop_and_enhance(pil_img):
    img = np.array(pil_img)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    # Crop dark borders
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        img = img[y:y+h, x:x+w]
        
    # Ben Graham enhancement
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), 10), -4, 128)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    return Image.fromarray(img)

infer_transform = transforms.Compose([
    transforms.Lambda(crop_and_enhance),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



# 3. ROBUST DATASET LOADER


In [ ]:

class SubmissionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = []
        self.transform = transform
        
        # os.walk guarantees we find every image, even if hidden in subfolders 
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp')):
                    self.image_paths.append(os.path.join(root, file))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        
        img_id = os.path.splitext(os.path.basename(img_path))[0]
        
        pil_img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            img_tensor = self.transform(pil_img)
            
        return img_tensor, img_id



# 4. MODEL BLUEPRINT FUNCTION

In [ ]:


def build_model():
    model = models.densenet121(weights=None)
    in_features = model.classifier.in_features
    
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 1)
    )
    return model



# 5. GENERATE SUBMISSION


In [ ]:


def create_submission():
    print("DEVICE:", DEVICE)

    
    print("\nScanning Test Directory...")
    test_dataset = SubmissionDataset(TEST_DIR, transform=infer_transform)
    
    if len(test_dataset) == 0:
        print(f"Error: No images found in {TEST_DIR}.")
        return
        
    print(f"Found {len(test_dataset)} images. Loading Dataloader...")
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    print("\nBuilding Model and Loading Weights...")
    model = build_model().to(DEVICE)
    
    if not os.path.exists(MODEL_WEIGHTS):
        print(f"Error: Weights file not found at {MODEL_WEIGHTS}")
        return
        
    # Load the trained weights
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=DEVICE))
    model.eval()
    
    predictions = []
    image_ids = []
    
    with torch.no_grad():
        # Using tqdm for a clean progress bar
        for images, img_ids in tqdm(test_loader, total=len(test_loader)):
            images = images.to(DEVICE)
            
            with autocast():
                preds = model(images)
            
            # Move predictions back to CPU
            preds_np = preds.cpu().squeeze(-1).numpy()
            
            predictions.extend(preds_np)
            image_ids.extend(img_ids)
            
    # Convert raw float predictions back to the 0-4 scale
    predictions = np.array(predictions)
    final_grades = np.clip(np.round(predictions), 0, 4).astype(int)
    
    # Format into the final DataFrame
    submission_df = pd.DataFrame({
        'id_code': image_ids,
        'diagnosis': final_grades
    })
    
    print(submission_df.head())
    
    submission_df.to_csv('/kaggle/working/submission.csv', index=False)
    print("submission.csv saved!")

if __name__ == '__main__':
    create_submission()